In [1]:
import sqlite3
import sys
sys.path.append('../src')
from loaders import load_assets

In [2]:
conn = sqlite3.connect('assets.sqlite')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS Assets;

CREATE TABLE Assets (
    asset_id    TEXT PRIMARY KEY,
    type    TEXT NOT NULL,
    description TEXT NOT NULL,
    location    TEXT NOT NULL,
    commissioning_date  TEXT NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('active', 'under repair', 'decommissioned')),
    linked_asset_id TEXT,
    FOREIGN KEY (linked_asset_id) REFERENCES Assets(asset_id)
);
''')

cur.execute('PRAGMA foreign_keys = ON')

surv_good, _ = load_assets('../data/assets.csv')

for line_num, row in surv_good:
    cur.execute(
        'INSERT INTO Assets (asset_id, type, description, location, commissioning_date, status, linked_asset_id) VALUES (?, ?, ?, ?, ?, ?, ?)',
        (row['asset_id'], row['type'], row['description'], row['location'],
         row['commissioning_date'], row['status'], None))

for line_num, row in surv_good:
    if row['linked_asset_id'] is not None:
        cur.execute(
            'UPDATE Assets SET linked_asset_id = ? WHERE asset_id = ?',
            (row['linked_asset_id'], row['asset_id']))

conn.commit()

In [3]:
cur.execute('SELECT asset_id, linked_asset_id FROM Assets')
for row in cur.fetchall():
    print(row)

('PMP-001', 'MTR-001')
('PMP-002', None)
('MTR-001', 'PMP-001')
('GEN-001', None)
('TRN-001', None)
('BLR-001', None)
('HVC-001', None)


In [4]:
try:
    cur.execute(
        'INSERT INTO Assets (asset_id, type, description, location, commissioning_date, status, linked_asset_id) VALUES (?, ?, ?, ?, ?, ?, ?)',
        ('GEN-099', 'generator', 'Test Generator', 'Generator Room', '2021-01-01', 'broken', None))
except sqlite3.IntegrityError as e:
    print(e)

try:
    cur.execute('UPDATE Assets SET linked_asset_id = ? WHERE asset_id = ?', ('PMP-999', 'MTR-001'))
except sqlite3.IntegrityError as e:
    print(e)

CHECK constraint failed: status IN ('active', 'under repair', 'decommissioned')
FOREIGN KEY constraint failed


In [5]:
cur.execute('SELECT asset_id, linked_asset_id FROM Assets')
for row in cur.fetchall():
    print(row)

('PMP-001', 'MTR-001')
('PMP-002', None)
('MTR-001', 'PMP-001')
('GEN-001', None)
('TRN-001', None)
('BLR-001', None)
('HVC-001', None)


In [6]:
conn.close()